# Assignment 2 — A hidden Markov model of your environment

You model the environment of your first assignment as a hidden Markov model: the **places** of your
ontology become the hidden states, and what your **sensors** report becomes the observations.

Sections 1–4 contain a complete example: the floor of lecture 8 (Entrance, Corridor, Seminar, Lab,
Coffee), with the matrices of slides 32, 36 and 37. **Replace the example with your own environment**,
run every cell from the top, and answer the questions in the cells marked **✎ Answer**.

Do not change the setup cell. To submit: `File → Download → Download .ipynb`, and name the file after the
surnames of the group members (e.g. `Rossi_Bianchi_Verdi.ipynb`).

In [ ]:
#@title Setup: installs hmmlearn and defines the helper functions. Run it first; do not change it.
!pip install -q hmmlearn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from hmmlearn.hmm import CategoricalHMM

def to_matrix(rows, row_labels, col_labels, name):
    """Turn {row: {column: probability}} into a matrix; missing entries are 0."""
    M = np.zeros((len(row_labels), len(col_labels)))
    for r, entries in rows.items():
        if r not in row_labels:
            raise ValueError(f"{name}: '{r}' is not in the list of rows")
        for c, p in entries.items():
            if c not in col_labels:
                raise ValueError(f"{name}: '{c}' (row '{r}') is not in the list of columns")
            M[row_labels.index(r), col_labels.index(c)] = p
    return M

def show_matrix(M, row_labels, col_labels, caption=""):
    df = pd.DataFrame(M, index=row_labels, columns=col_labels)
    display(df.style.format("{:.2f}").set_caption(caption)
              .background_gradient(cmap="Blues", vmin=0, vmax=1))

def show_graph(states, adjacent):
    G = nx.Graph(); G.add_nodes_from(states); G.add_edges_from(adjacent)
    pos = nx.spring_layout(G, seed=3)
    plt.figure(figsize=(6, 4))
    nx.draw_networkx(G, pos, node_color="#C3D8EC", node_size=2600, font_size=10, edge_color="#2C6395", width=2)
    plt.margins(0.15); plt.axis("off"); plt.show()

def check(name, M, row_labels, ok):
    sums = M.sum(axis=1)
    for r, s in zip(row_labels, sums):
        if abs(s - 1) > 1e-6:
            print(f"✗ {name}: row '{r}' adds up to {s:.3f}, not 1"); ok = False
    if (M < 0).any():
        print(f"✗ {name}: negative entries"); ok = False
    return ok

def check_model():
    ok = True
    ok = check("transition matrix", A, STATES, ok)
    ok = check("observation matrix", B, STATES, ok)
    if abs(PI.sum() - 1) > 1e-6:
        print(f"✗ start distribution adds up to {PI.sum():.3f}, not 1"); ok = False
    near = {(a, b) for a, b in ADJACENT} | {(b, a) for a, b in ADJACENT}
    for i, a in enumerate(STATES):
        if A[i, i] == 0:
            print(f"✗ '{a}' has no probability of staying put"); ok = False
        for j, b in enumerate(STATES):
            if i != j and A[i, j] > 0 and (a, b) not in near:
                print(f"✗ '{a}' → '{b}' is not 0, but the two places are not adjacent"); ok = False
            if i != j and A[i, j] == 0 and (a, b) in near:
                print(f"! '{a}' → '{b}' is 0, but the two places are adjacent: is that intended?")
    print("✓ the model is consistent" if ok else "Fix the lines marked ✗ before going on.")

def make_model(A_matrix):
    m = CategoricalHMM(n_components=len(STATES), init_params="", params="")
    m.n_features = len(OBSERVATIONS)
    m.startprob_, m.transmat_, m.emissionprob_ = PI, A_matrix, B
    return m

def walk(model, steps, seed):
    X, Z = model.sample(steps, random_state=seed)
    return X.ravel(), Z

def decode(model, readings):
    X = np.array([OBSERVATIONS.index(r) for r in readings]).reshape(-1, 1)
    return [STATES[s] for s in model.decode(X, algorithm="viterbi")[1]]

def show_walk(model, steps, seed):
    obs, true = walk(model, steps, seed)
    readings = [OBSERVATIONS[o] for o in obs]
    decoded = decode(model, readings)
    guess = [STATES[int(np.argmax(B[:, o]))] for o in obs]   # the place that best explains each reading alone
    table = pd.DataFrame({"true place": [STATES[s] for s in true], "reading": readings,
                          "reading alone": guess, "decoded (Viterbi)": decoded})
    acc_d = np.mean([d == STATES[s] for d, s in zip(decoded, true)])
    acc_g = np.mean([g == STATES[s] for g, s in zip(guess, true)])
    print(f"Correct places: decoded {acc_d:.0%}, reading alone {acc_g:.0%}  ({steps} steps)")
    display(table.style.apply(lambda row: ["background-color: #FFF2B3" if row["decoded (Viterbi)"] != row["true place"] else ""
                                           for _ in row], axis=1))

def classify(models, recordings, steps, seed):
    rng = np.random.RandomState(seed)
    names = list(models)
    counts = pd.DataFrame(0, index=[f"is a {n}" for n in names], columns=[f"classified as {n}" for n in names])
    for true_name in names:
        for _ in range(recordings):
            X, _ = models[true_name].sample(steps, random_state=rng)
            scores = [models[n].score(X) for n in names]          # forward algorithm: log-likelihood
            counts.loc[f"is a {true_name}", f"classified as {names[int(np.argmax(scores))]}"] += 1
    display(counts)

## 1. Places and connections

List the places of your environment (at least five) and the pairs of places that are directly connected.
Use the place individuals of your ontology and its adjacency property (for example `adjacentTo`).
Also state the time step: the time between two readings.

In [ ]:
STATES = ["Entrance", "Corridor", "Seminar", "Lab", "Coffee"]

ADJACENT = [("Entrance", "Corridor"), ("Corridor", "Seminar"), ("Corridor", "Coffee"),
            ("Seminar", "Lab"), ("Coffee", "Lab")]

TIME_STEP = "5 seconds"

show_graph(STATES, ADJACENT)

**✎ Answer.** Which individuals and which property of your ontology do these places and connections come
from? Did you have to add or change anything with respect to the ontology, and why?

## 2. Transition matrix

For each place, the probability of being in each place one time step later. Write only the entries that
are not zero: a missing entry is 0. Every row must add up to one, staying put included. A move between
two places that are not adjacent must be 0.

In [ ]:
TRANSITIONS = {
    "Entrance": {"Entrance": 0.60, "Corridor": 0.40},
    "Corridor": {"Entrance": 0.15, "Corridor": 0.40, "Seminar": 0.25, "Coffee": 0.20},
    "Seminar":  {"Corridor": 0.10, "Seminar": 0.85, "Lab": 0.05},
    "Lab":      {"Seminar": 0.05, "Lab": 0.85, "Coffee": 0.10},
    "Coffee":   {"Corridor": 0.25, "Lab": 0.15, "Coffee": 0.60},
}
A = to_matrix(TRANSITIONS, STATES, STATES, "TRANSITIONS")
show_matrix(A, STATES, STATES, "from (rows) → to (columns), one time step later")

**✎ Answer.** Justify the values on the diagonal given your time step, and one value off the diagonal.

## 3. Observation matrix

At each time step the system receives **one** observation. With a location system it is the place the
system reports. With sensors installed only in some places it is the sensor that fired (for example
`"pir_ward"` or `"door_toilet"`), or `"none"`. List all possible observations, then give, for each place,
the probability of each observation. Every row must add up to one.

In [ ]:
OBSERVATIONS = ["Entrance", "Corridor", "Seminar", "Lab", "Coffee", "Unknown"]

EMISSIONS = {
    "Entrance": {"Entrance": 0.85, "Corridor": 0.05, "Unknown": 0.10},
    "Corridor": {"Corridor": 0.70, "Coffee": 0.10, "Unknown": 0.20},
    "Seminar":  {"Seminar": 0.85, "Lab": 0.05, "Unknown": 0.10},
    "Lab":      {"Seminar": 0.05, "Lab": 0.85, "Unknown": 0.10},
    "Coffee":   {"Corridor": 0.25, "Coffee": 0.55, "Unknown": 0.20},
}
B = to_matrix(EMISSIONS, STATES, OBSERVATIONS, "EMISSIONS")
show_matrix(B, STATES, OBSERVATIONS, "true place (rows) → what the system receives (columns)")

**✎ Answer.** Which errors or limits of your sensors does this matrix represent? Which places produce
observations that are hard to tell apart?

## 4. Start distribution and check

The probability of each place at the first time step. Then run the check: it reports rows that do not add
up to one and moves between places that are not adjacent.

In [ ]:
START = {"Entrance": 0.50, "Corridor": 0.20, "Seminar": 0.15, "Lab": 0.10, "Coffee": 0.05}
PI = to_matrix({"start": START}, ["start"], STATES, "START")[0]

check_model()

## 5. Decoding a walk

The notebook generates a walk with your model, together with the observations it produces, and decodes the
observations with the Viterbi algorithm. The table compares the true places, the place that best explains
each observation on its own, and the decoded sequence; rows where the decoded place is wrong are highlighted.

In [ ]:
model = make_model(A)
show_walk(model, steps=40, seed=1)

In [ ]:
# Decode a sequence of observations written by hand (the example is slide 41 of lecture 8).
decode(model, ["Seminar", "Lab", "Seminar"])

**✎ Answer.** Where does the decoded sequence differ from the true one, and why? Compare it with the
column "reading alone". Then write, in the cell above, a short sequence of observations that your model
decodes differently from what the observations alone suggest, and explain the result.

## 6. Two kinds of people

Write a second transition matrix for a different kind of person in your environment (in the example, a
visitor instead of somebody who works there), with the same places and the same sensors. The notebook
generates recordings from both models and classifies each recording with the forward algorithm: the model
under which the recording is more probable wins.

In [ ]:
KIND_1, KIND_2 = "person who works here", "visitor"

TRANSITIONS_2 = {
    "Entrance": {"Entrance": 0.55, "Corridor": 0.45},
    "Corridor": {"Entrance": 0.15, "Corridor": 0.45, "Seminar": 0.20, "Coffee": 0.20},
    "Seminar":  {"Corridor": 0.25, "Seminar": 0.60, "Lab": 0.15},
    "Lab":      {"Seminar": 0.25, "Lab": 0.50, "Coffee": 0.25},
    "Coffee":   {"Corridor": 0.30, "Lab": 0.20, "Coffee": 0.50},
}
A2 = to_matrix(TRANSITIONS_2, STATES, STATES, "TRANSITIONS_2")
show_matrix(A2, STATES, STATES, f"transition matrix of a {KIND_2}")

classify({KIND_1: make_model(A), KIND_2: make_model(A2)}, recordings=50, steps=12, seed=2)

**✎ Answer.** Which rows of the two matrices differ, and why? How often is a recording classified wrongly,
and in which direction? What would make the two kinds easier to tell apart?

## Before you submit

- Sections 1–4 describe **your** environment, not the example, and the check prints ✓.
- Every cell marked **✎ Answer** has an answer.
- The notebook runs from top to bottom without errors (`Runtime → Restart session and run all`).